# 05 · Memory planning and VM lowering

After scheduling, the remaining passes turn a functional dataflow graph into instructions for TVM's
virtual machine. In order, they:
- make every output buffer explicit;
- plan memory statically;
- free tensors early;
- lower shape computations;
- attach the external modules.

This notebook follows those passes on the tiny model, then compares the final bytecode with the real
artifact's.

**Needs:** CUDA. About a minute.

In [1]:
from _helpers import *
bootstrap()

repo root: fast-dllm-relax/  (cwd set)  tvm 0.24.dev0  torch 2.10.0+cu128


PosixPath('<repo>')

In [2]:
import tvm
from tvm import relax
from tvm.s_tir import dlight

model, mod, named = export_model(tiny_config(tied=True), dtype="float16")
dev = tvm.cuda(0)
target = tvm.target.Target.from_device(dev)
pipeline = relax.get_pipeline("opt_fast_dllm_llm_cublas", use_cutlass=True, verbose=False)

rec = make_recorder()
lowered = lower(mod, target, pipeline, recorder=rec)
pass_table(rec, max_depth=0, cols=("relax_fns", "tir_fns", "calls", "extern_calls"))
FN = "forward_cached"

[16:56:06] <repo>/3rdparty/tvm/include/tvm/topi/transform.h:1253: Warning: Fast mode segfaults when there are out-of-bounds indices. Make sure input indices are in bound


#     pass                    ms         relax_fns  tir_fns  calls  extern_calls
----  ----------------------  ---------  ---------  -------  -----  ------------
0     FuseOpsByPattern        84.9       17         0        1416   12          
1     FuseOpsByPattern        507        18         0        1418   12          
2     RunCodegen              1.026e+04  10         0        1353   77          
4     FuseTransposeMatmul     17.9       10         0        1353   77          
5     LegalizeOps             281.1      10         125      1329   77          
6     AnnotateTIROpPattern    0.4        10         125      1329   77          
3847  FoldConstant            1248       10         125      1265   77          
3848  FuseOps                 25.5       58         125      736    77          
3858  FuseTIR                 87.2       10         69       485    77          
3859  DeadCodeElimination     4.2        10         69       485    77          
3860  ApplyDefaultSchedule  

## `ToNonDataflow` and `CallTIRRewrite`: explicit output buffers

`R.call_tir(kernel, args, out_sinfo)` is pure: it returns a new tensor. `CallTIRRewrite` splits it
into "allocate the output" + "call the kernel with the output as an argument" (destination-passing
style). Before and after:

In [3]:
diff_ir(rec.before("CallTIRRewrite"), rec.after("CallTIRRewrite"), FN, max_lines=30)

--- before
+++ after
@@ -22,11 +22,27 @@
     model_layers_1_post_attention_layernorm_weight5: R.Tensor((256,), dtype="float16") = packed_params[14]
     model_norm_weight5: R.Tensor((256,), dtype="float16") = packed_params[15]
-    lv180 = R.call_tir(fused_strided_slice_squeeze, (cached_k,), out_sinfo=R.Tensor((1, 2, prefix_len, 128), dtype="float16"))
-    lv181 = R.call_tir(fused_strided_slice_squeeze, (cached_v,), out_sinfo=R.Tensor((1, 2, prefix_len, 128), dtype="float16"))
-    astype49 = R.call_tir(cast, (active_embed,), out_sinfo=R.Tensor((1, block_len, 256), dtype="float32"))
-    square14 = R.call_tir(tir_square, (astype49,), out_sinfo=R.Tensor((1, block_len, 256), dtype="float32"))
-    lv182 = R.call_tir(fused_mean_add_tir_rsqrt_multiply_cast1_multiply1, (square14, astype49, model_layers_0_input_layernorm_weight5), out_sinfo=R.Tensor((1, block_len, 256), dtype="float16"))
-    lv4 = R.call_dps_packed("fused_relax_permute_dims_relax_matmul_relax_add_cublas", (model_layers_0_

## `StaticPlanBlockMemory`: reuse buffers whose lifetimes do not overlap

The planner changes each `alloc_tensor` into an `alloc_tensor` out of a shared `alloc_storage` token,
so later tensors reuse storage that earlier ones released. It can only plan sizes it knows
statically. Allocations whose size depends on a symbolic dim with no upper bound (`prefix_len`,
`seq_len`) stay as runtime allocations.

In [4]:
rows = []
for fn in ["get_logits", "forward_hidden", "forward_cached", FN, "forward_block_partial_rp8"]:
    before = op_counts(rec.before("StaticPlanBlockMemory"), fn)
    after = op_counts(rec.after("StaticPlanBlockMemory"), fn)
    rows.append({"function": fn,
                 "allocs before": before.get("relax.builtin.alloc_tensor", 0),
                 "planned storages": after.get("relax.memory.alloc_storage", 0),
                 "tensors in planned storage": after.get("relax.memory.alloc_tensor", 0),
                 "still runtime-allocated": after.get("relax.builtin.alloc_tensor", 0)})
print_table(rows)

function                   allocs before  planned storages  tensors in planned storage  still runtime-allocated
-------------------------  -------------  ----------------  --------------------------  -----------------------
get_logits                 2              1                 1                           1                      
forward_hidden             56             14                55                          1                      
forward_cached             60             18                57                          3                      
forward_cached_block       60             19                57                          3                      
forward_block_partial_rp8  63             17                59                          4                      


In [5]:
show_ir(rec.after("StaticPlanBlockMemory"), FN, grep=r"alloc_storage|builtin\.alloc_tensor", max_lines=20)

    storage101: R.Object = R.memory.alloc_storage(R.shape([512 * prefix_len]), R.prim_value(0), R.str("global"), R.dtype("float16"))
    ...
    storage102: R.Object = R.memory.alloc_storage(R.shape([512 * prefix_len]), R.prim_value(0), R.str("global"), R.dtype("float16"))
    ...
    storage103: R.Object = R.memory.alloc_storage(R.shape([1024 * block_len]), R.prim_value(0), R.str("global"), R.dtype("float32"))
    ...
    storage104: R.Object = R.memory.alloc_storage(R.shape([1024 * block_len]), R.prim_value(0), R.str("global"), R.dtype("float32"))
    ...
    storage105: R.Object = R.memory.alloc_storage(R.shape([512 * block_len]), R.prim_value(0), R.str("global"), R.dtype("float16"))
    ...
    storage106: R.Object = R.memory.alloc_storage(R.shape([2048 * block_len]), R.prim_value(0), R.str("global"), R.dtype("float16"))
    ...
    storage107: R.Object = R.memory.alloc_storage(R.shape([1024 * block_len]), R.prim_value(0), R.str("global"), R.dtype("float16"))
    ...
    storage108

## `RewriteCUDAGraph`: on only when asked

This pass can capture static regions of a function into a CUDA graph, removing the per-kernel
launch overhead. But it **only runs when the `PassContext` config `relax.backend.use_cuda_graph` is
true** (`src/relax/transform/rewrite_cuda_graph.cc`), and the production export never sets it. So in
the shipped artifacts it does nothing:

In [6]:
diff_ir(rec.before("RewriteCUDAGraph"), rec.after("RewriteCUDAGraph"), max_lines=5)

(no change)


Here is what it would do with the config set:

In [7]:
rec_g = make_recorder()
lowered_g = lower(mod, target, pipeline, recorder=rec_g, config={"relax.backend.use_cuda_graph": True})
after_g = rec_g.after("RewriteCUDAGraph")
captured = find_functions(after_g, r"cuda_graph")
print("new functions:", captured)
rows = []
for fn in ["get_logits", "forward_hidden", FN, "forward_block_partial_rp8"]:
    c = op_counts(after_g, fn)
    rows.append({"function": fn, "cuda-graph builtin calls": sum(v for k, v in c.items() if "cuda_graph" in k)})
print_table(rows)

[17:08:50] <repo>/3rdparty/tvm/include/tvm/topi/transform.h:1253: Warning: Fast mode segfaults when there are out-of-bounds indices. Make sure input indices are in bound


new functions: ['cuda_graph_alloc', 'forward_block_partial_rp0_cuda_graph_capture', 'forward_block_partial_rp0_cuda_graph_capture1', 'forward_block_partial_rp0_cuda_graph_capture2', 'forward_block_partial_rp0_cuda_graph_capture3', 'forward_block_partial_rp0_cuda_graph_capture4', 'forward_block_partial_rp0_cuda_graph_capture5']
function                   cuda-graph builtin calls
-------------------------  ------------------------
get_logits                 0                       
forward_hidden             0                       
forward_cached_block       0                       
forward_block_partial_rp8  8                       


Capture needs static shapes and static allocations, so only the parts of a function that do not
depend on `prefix_len`/`seq_len` qualify. The attention callback cannot be captured either, because
it runs in Python. How much of a decode step this would capture is an open experiment (research 001,
open question 3). The answer above is the starting point.

## `KillAfterLastUse`, `VMShapeLower`: VM bookkeeping

`KillAfterLastUse` inserts frees right after each tensor's last use, so memory is released during a
call rather than at its end. `VMShapeLower` turns symbolic-shape logic into VM builtins
(`match_shape`, a shape heap), so the VM can check inputs and compute output sizes at runtime.

In [8]:
print_table([{"op": k, "count": v} for k, v in op_counts(lowered, FN).most_common(15)])

op                                                        count
--------------------------------------------------------  -----
relax.vm.kill_object                                      107  
extern:vm.builtin.make_shape                              96   
relax.vm.alloc_tensor                                     60   
relax.vm.alloc_storage                                    22   
extern:vm.builtin.reshape                                 8    
extern:vm.builtin.match_shape                             6    
extern:vm.builtin.call_tir_dyn                            6    
global:tir_square                                         5    
global:fused_mean_add_tir_rsqrt_multiply_cast1_multiply1  5    
extern:vm.builtin.check_tensor_info                       4    
global:cast                                               4    
global:concatenate14                                      4    
extern:fused_relax_permute_dims_relax_matmul_cublas       4    
global:add4                             

## The executable: what one call runs

In [9]:
ex = build(lowered, target)
fns = vm_functions(ex)
rows = []
for fn in sorted(f for f in fns if not f.startswith("vm.")):
    c = call_summary(ex, fn)
    rows.append({"function": fn, "instructions": len(fns[fn]), **{k: c.get(k, 0) for k in
                 ["cuBLAS", "CUTLASS", "attention callback", "TIR kernel", "memory builtin", "other builtin"]}})
print_table(rows)

function                    instructions  cuBLAS  CUTLASS  attention callback  TIR kernel  memory builtin  other builtin
--------------------------  ------------  ------  -------  ------------------  ----------  --------------  -------------
build_prefix_cache          245           5       1        0                   26          123             89           
embed                       15            0       0        0                   2           5               7            
forward_block_partial_rp0   304           8       0        2                   48          188             57           
forward_block_partial_rp16  304           8       0        2                   48          188             57           
forward_block_partial_rp24  304           8       0        2                   48          188             57           
forward_block_partial_rp8   304           8       0        2                   48          188             57           
forward_cached              385 

In [10]:
bytecode(ex, FN, grep=r"cublas|cutlass|flash_attn|fused_", max_lines=25)

  call  fused_strided_slice_squeeze in: %1, %24      dst: %void
  call  fused_strided_slice_squeeze in: %2, %28      dst: %void
  call  fused_mean_add_tir_rsqrt_multiply_cast1_multiply1 in: %36, %32, %11, %40 dst: %void
  call  fused_relax_permute_dims_relax_matmul_relax_add_cublas in: %6, %40, %7, %44 dst: %45
  call  vm.builtin.call_tir_dyn in: f[fused_arange_cast10_add10_expand_dims_multiply3_concatenate], %3, c[7], %66, %67 dst: %void
  call  fused_tir_cos_expand_dims2_expand_dims3_cast3 in: %66, %71     dst: %void
  call  fused_tir_sin_expand_dims2_expand_dims3_cast3 in: %66, %75     dst: %void
  call  fused_multiply4_split1_multiply5_concatenate1_multiply4_add1 in: %58, %71, %75, %79 dst: %void
  call  fused_multiply6_split2_multiply7_concatenate2_multiply6_add2_transpose in: %60, %71, %75, %83 dst: %void
  call  fast_dllm.flash_attn_bf16 in: %79, %97, %100 dst: %102
  call  fused_relax_permute_dims_relax_matmul_cublas in: %8, %104, %106 dst: %107
  call  fused_mean_add_tir_rsqrt

With 2 layers, `forward_cached` makes 2 × 4 cuBLAS calls (qkv, o_proj, gate_up, down) and 2
attention callbacks. The real 28-layer artifact should show 112 and 28:

In [11]:
from pathlib import Path
from fast_dllm_relax.export_utils import module_kind_counts
so = Path("artifacts/fast_dllm_relax_bf16/fast_dllm_relax_nn.so")
real = load_executable(so)
print("module kinds in the artifact:", module_kind_counts(real.mod))
rows = []
for fn in ["embed", "get_logits", "forward_hidden", "build_prefix_cache", "forward_cached", FN, "forward_block_partial_rp8"]:
    c = call_summary(real, fn)
    rows.append({"function": fn, **{k: c.get(k, 0) for k in ["cuBLAS", "CUTLASS", "attention callback", "TIR kernel"]}})
print_table(rows)

module kinds in the artifact: {'relax.VMExecutable': 1, 'const_loader': 1, 'cublas_json': 9, 'library': 1, 'cuda': 1}
function                   cuBLAS  CUTLASS  attention callback  TIR kernel
-------------------------  ------  -------  ------------------  ----------
embed                      0       0        0                   2         
get_logits                 1       0        0                   2         
forward_hidden             112     0        0                   535       
build_prefix_cache         109     0        0                   546       
forward_cached             112     0        28                  535       
forward_cached_block       112     0        28                  537       
forward_block_partial_rp8  112     0        28                  594       


## Takeaways
- After scheduling, the pipeline makes memory explicit (`CallTIRRewrite`), plans it
  (`StaticPlanBlockMemory`) and frees it early (`KillAfterLastUse`). The KV cache is not among those
  allocations: it is a preallocated input written in place, so the cached forwards' own buffers depend
  only on the block length, and the pooled allocator reuses them step after step.
- `RewriteCUDAGraph` is present in the pipeline but **inactive**. Turning it on is a config flag plus
  a measurement.
- The bytecode is a flat list of calls. Counting callees is a reliable way to check what an artifact
  actually offloads.
- Next: **06 · Runtime and decode loop**.